# Retrieval-Augmented Generation

**Companion lesson:** https://ml-viz.vercel.app/courses/building-with-llms/04-retrieval-augmented-generation

A from-scratch, runnable implementation of the concepts in the lesson — pure NumPy, no API keys required.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## A minimal RAG pipeline

We reuse the toy embedder, add **chunking**, retrieve the top-k chunks for a query, assemble them into a context prompt, and run a mock `generate` that answers *only* from the retrieved context (extractive). No API key needed — the point is the pipeline shape.

In [ ]:
DOC = (
    'Our return policy allows refunds within 30 days of purchase. '
    'To request a refund, email support with your order number. '
    'Standard shipping takes 3 to 5 business days. '
    'International shipping can take up to 2 weeks. '
    'You can reset your password from the account settings page.'
)

def chunk(text, size=8, overlap=2):
    words = text.split()
    out, i = [], 0
    while i < len(words):
        out.append(' '.join(words[i:i + size]))
        i += size - overlap
    return out

chunks = chunk(DOC)
for c in chunks:
    print('-', c)

In [ ]:
def tokenize(s):
    return re.findall(r'[a-z]+', s.lower())

vocab = sorted({w for c in chunks for w in tokenize(c)})
index = {w: i for i, w in enumerate(vocab)}
def embed(text):
    v = np.zeros(len(vocab))
    for w in tokenize(text):
        if w in index:
            v[index[w]] += 1.0
    return v
def cosine(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return 0.0 if na == 0 or nb == 0 else float(a @ b / (na * nb))

chunk_vecs = np.array([embed(c) for c in chunks])

def retrieve(query, k=2):
    q = embed(query)
    sims = [cosine(q, c) for c in chunk_vecs]
    return [chunks[i] for i in np.argsort(sims)[::-1][:k]]

def generate(query, k=2):
    context = retrieve(query, k)
    prompt = 'Answer using ONLY this context:\n' + '\n'.join(context) + f'\nQ: {query}\nA:'
    print(prompt)
    print('\n[mock answer would be grounded in the context above]')

generate('how long do refunds take to request')

Increase `k` and watch off-topic chunks (shipping, passwords) leak into the context — the precision/coverage trade-off from the lesson.

## ✏️ Your turn

Implement `recall_at_k`: given the indices of the truly relevant chunks and the retrieved indices, return the fraction of relevant chunks that were retrieved.

In [ ]:
def recall_at_k(relevant, retrieved):
    # TODO(you): fraction of `relevant` indices that appear in `retrieved`.
    return 0.0

assert recall_at_k({0, 1}, [0, 3]) == 0.5
assert recall_at_k({2}, [2, 0, 1]) == 1.0
print('passed ✓')

<details><summary>Solution</summary>

```python
def recall_at_k(relevant, retrieved):
    relevant = set(relevant)
    hits = len(relevant & set(retrieved))
    return hits / len(relevant)
```

</details>